In [ ]:
#!/usr/bin/env python3
# ============================================================
# Empirical OS-bridge lower bound:
#   theta_hat ≈ inf_f  <f,(I-K)f> / <f,(I-P)f>
#
# This script is FULLY STANDALONE and RUNS as-is.
#
# What it does:
#   - Estimates Dirichlet forms E_K(f)=<f,(I-K)f> and E_P(f)=<f,(I-P)f>
#     via Monte Carlo pairs (X, Y_K) with Y_K ~ K(X), and (X, Y_P) with Y_P ~ P(X)
#   - Sweeps a test family of functions f, returns min ratio = theta_hat.
#
# What it does NOT do:
#   - It does NOT implement your real lattice OS strip kernel K_a.
#   - It provides a drop-in interface: plug in your K_a sampler and your P_a sampler.
#
# Default demo model (so it runs):
#   - State space: SU(2)^n_links as unit quaternions (Haar stationary).
#   - K: "strip-like" step = smaller Brownian heat-kernel step
#   - P: "diffusion-like" step = larger Brownian heat-kernel step
#   (Pure diagnostic harness.)
# ============================================================

import argparse, math
import numpy as np

# ---------------------------
# SU(2) as unit quaternions
# ---------------------------

def su2_haar_quaternion(rng: np.random.Generator, shape):
    """
    Haar on SU(2) via uniform on S^3.
    Returns array with last dim = 4 (w,x,y,z) and norm 1.
    """
    x = rng.normal(size=(*shape, 4)).astype(np.float64)
    x /= (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-30)
    # Fix sign to reduce double cover nuisance (optional)
    # Negate quaternions where the 'w' component is negative.
    mask = x[..., 0] < 0
    x[mask] = -x[mask]
    return x

def quat_mul(a, b):
    """Quaternion product. a,b (...,4) in (w,x,y,z)."""
    aw, ax, ay, az = np.moveaxis(a, -1, 0)
    bw, bx, by, bz = np.moveaxis(b, -1, 0)
    w = aw*bw - ax*bx - ay*by - az*bz
    x = aw*bx + ax*bw + ay*bz - az*by
    y = aw*by - ax*bz + ay*bw + az*bx
    z = aw*bz + ax*by - ay*bx + az*bw
    return np.stack([w, x, y, z], axis=-1)

def quat_norm(a):
    return np.linalg.norm(a, axis=-1, keepdims=True)

def quat_unit(a):
    return a / quat_norm(a)

def su2_brownian_step(rng: np.random.Generator, U: np.ndarray, t: float):
    """
    Heat-kernel proxy step on SU(2): multiply by small random group element exp(sqrt(t)*X)
    using quaternion approximation: sample small Lie algebra vector v~N(0,t I3),
    map to unit quaternion q = (cos(|v|), sin(|v|) v/|v|) with |v| in R.
    """
    # v in R^3
    v = rng.normal(size=U.shape[:-1] + (3,)) * math.sqrt(max(t, 0.0))
    r = np.linalg.norm(v, axis=-1, keepdims=True)
    # avoid 0/0
    r_safe = np.where(r == 0, 1.0, r)
    axis = v / r_safe
    # SU(2) parameterization: angle = r
    w = np.cos(r)[..., 0]
    s = np.sin(r)[..., 0]
    # Correctly unpack the x, y, z components
    scaled_axis = s[..., None] * axis
    x = scaled_axis[..., 0]
    y = scaled_axis[..., 1]
    z = scaled_axis[..., 2]
    q = np.stack([w, x, y, z], axis=-1)
    q = quat_unit(q)
    return quat_unit(quat_mul(q, U))

# ---------------------------
# Stationary sampler and kernels
# Replace these three with your real YM objects.
# ---------------------------

def sample_nu(rng: np.random.Generator, n_samples: int, n_links: int):
    """
    Stationary sampler for boundary measure nu.
    DEMO: Haar on SU(2)^n_links.
    Replace with your boundary sampler if you have one.
    """
    return su2_haar_quaternion(rng, shape=(n_samples, n_links))

def step_K(rng: np.random.Generator, X: np.ndarray, tK: float):
    """
    One-step sampler Y ~ K(X).
    DEMO: smaller Brownian step.
    Replace with: OS strip kernel sampler.
    """
    return su2_brownian_step(rng, X, t=tK)

def step_P(rng: np.random.Generator, X: np.ndarray, tP: float):
    """
    One-step sampler Y ~ P(X).
    DEMO: larger Brownian step.
    Replace with: configuration diffusion step sampler.
    """
    return su2_brownian_step(rng, X, t=tP)

# ---------------------------
# Test function family f
# ---------------------------

class FeatureFamily:
    """
    f_theta(X) = sum_{links} <A_ell, quat(X_ell)> + b
    with A_ell in R^4.
    Mean-centering is done empirically per batch so f has ~zero mean under nu samples.
    """
    def __init__(self, rng: np.random.Generator, n_links: int, n_funcs: int, scale: float = 1.0):
        self.n_links = n_links
        self.n_funcs = n_funcs
        self.A = rng.normal(size=(n_funcs, n_links, 4)) * scale
        self.b = rng.normal(size=(n_funcs,)) * (0.1 * scale)

    def eval(self, X: np.ndarray) -> np.ndarray:
        """
        X: (N, n_links, 4)
        returns F: (n_funcs, N)
        """
        # (n_funcs, N, n_links)
        proj = np.einsum("f l d, n l d -> f n l", self.A, X)
        val = np.sum(proj, axis=-1) + self.b[:, None]
        return val

# ---------------------------
# Dirichlet form estimators
# ---------------------------

def dirichlet_from_pairs(fX: np.ndarray, fY: np.ndarray) -> np.ndarray:
    """
    For reversible Markov kernel K with stationary nu:
      <f, (I-K) f> = 1/2 E[(f(X)-f(Y))^2]  with X~nu, Y~K(X)
    Using samples, return per-function estimates.
    Inputs:
      fX, fY: (n_funcs, N)
    Returns:
      E: (n_funcs,)
    """
    d = fX - fY
    return 0.5 * np.mean(d * d, axis=1)

def variance_empirical(fX: np.ndarray) -> np.ndarray:
    """
    fX: (n_funcs, N) with mean not necessarily zero.
    returns Var estimates per function.
    """
    m = np.mean(fX, axis=1, keepdims=True)
    return np.mean((fX - m) ** 2, axis=1)

# ---------------------------
# Main experiment
# ---------------------------

def estimate_theta_hat(
    seed: int,
    n_links: int,
    n_samples: int,
    n_funcs: int,
    tK: float,
    tP: float,
    scale: float,
    eps: float,
):
    rng = np.random.default_rng(seed)

    # Sample X ~ nu
    X = sample_nu(rng, n_samples=n_samples, n_links=n_links)

    # Sample Y_K ~ K(X), Y_P ~ P(X)
    YK = step_K(rng, X, tK=tK)
    YP = step_P(rng, X, tP=tP)

    # Build test functions and evaluate
    fam = FeatureFamily(rng, n_links=n_links, n_funcs=n_funcs, scale=scale)
    fX = fam.eval(X)
    fYK = fam.eval(YK)
    fYP = fam.eval(YP)

    # Empirical mean-centering on X samples (restrict to mean-zero subspace)
    fXc = fX - np.mean(fX, axis=1, keepdims=True)
    fYKc = fYK - np.mean(fYK, axis=1, keepdims=True)
    fYPc = fYP - np.mean(fYP, axis=1, keepdims=True)

    # Compute Dirichlet forms and ratio
    EK = dirichlet_from_pairs(fXc, fYKc)  # <f,(I-K)f>
    EP = dirichlet_from_pairs(fXc, fYPc)  # <f,(I-P)f>

    # Filter out degenerate EP
    good = EP > eps
    ratios = np.full((n_funcs,), np.inf)
    ratios[good] = EK[good] / EP[good]

    theta_hat = float(np.min(ratios[np.isfinite(ratios)])) if np.any(np.isfinite(ratios)) else float("nan")

    # Also report diagnostic: implied PI constant for K on this family:
    # Var <= (1/c) E_K  => c_hat = min(E_K/Var)
    Var = variance_empirical(fXc)
    good2 = Var > eps
    c_hat = float(np.min((EK[good2] / Var[good2]))) if np.any(good2) else float("nan")

    # Return full stats
    return {
        "theta_hat_min_ratio_EK_over_EP": theta_hat,
        "c_hat_min_ratio_EK_over_Var": c_hat,
        "EK_min": float(np.min(EK)),
        "EK_med": float(np.median(EK)),
        "EP_min": float(np.min(EP)),
        "EP_med": float(np.median(EP)),
        "n_funcs": int(n_funcs),
        "n_samples": int(n_samples),
        "n_links": int(n_links),
        "tK": float(tK),
        "tP": float(tP),
    }

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--n_links", type=int, default=64, help="Boundary degrees of freedom (replace with your boundary size).")
    ap.add_argument("--n_samples", type=int, default=20000)
    ap.add_argument("--n_funcs", type=int, default=256)
    ap.add_argument("--tK", type=float, default=0.02, help="DEMO: strip step strength")
    ap.add_argument("--tP", type=float, default=0.05, help="DEMO: diffusion step strength")
    ap.add_argument("--scale", type=float, default=1.0, help="Test function weight scale")
    ap.add_argument("--eps", type=float, default=1e-12)
    args = ap.parse_args([])

    out = estimate_theta_hat(
        seed=args.seed,
        n_links=args.n_links,
        n_samples=args.n_samples,
        n_funcs=args.n_funcs,
        tK=args.tK,
        tP=args.tP,
        scale=args.scale,
        eps=args.eps,
    )

    # Print in a stable, grep-friendly format
    print("=== BRIDGE LOWER-BOUND ESTIMATE (TEST FAMILY) ===")
    for k in [
        "theta_hat_min_ratio_EK_over_EP",
        "c_hat_min_ratio_EK_over_Var",
        "EK_min","EK_med",
        "EP_min","EP_med",
        "n_funcs","n_samples","n_links",
        "tK","tP",
    ]:
        print(f"{k:32s}: {out[k]}")

if __name__ == "__main__":
    main()

=== BRIDGE LOWER-BOUND ESTIMATE (TEST FAMILY) ===
theta_hat_min_ratio_EK_over_EP  : 0.39135553726598066
c_hat_min_ratio_EK_over_Var     : 0.032769145967673205
EK_min                          : 1.473467974768137
EK_med                          : 1.8860156130805854
EP_min                          : 3.5947578973683316
EP_med                          : 4.637004213143814
n_funcs                         : 256
n_samples                       : 20000
n_links                         : 64
tK                              : 0.02
tP                              : 0.05


In [ ]:
#!/usr/bin/env python3
# ============================================================
# FINAL BOSS: Empirical OS-bridge lower bound
# (Jupyter / Colab safe — NO argparse)
#
# Computes true Rayleigh quotients over LINEAR COMBINATIONS
# of features:
#
#   theta_hat = inf_w  (w^T A_K w) / (w^T A_P w)
#   c_hat     = inf_w  (w^T A_K w) / (w^T V   w)
#
# Drop-in hooks (REPLACE ONLY THESE when ready):
#   sample_nu
#   step_K
#   step_P
# ============================================================

import math
import numpy as np

# ===========================
# CONFIG (edit here)
# ===========================

CONFIG = dict(
    seed=0,
    n_links=64,
    n_samples=20000,
    m_feats=256,
    tK=0.02,
    tP=0.05,
    scale=1.0,
    ridge=1e-6,
    chunks=8,
)

# ===========================
# SU(2) demo (quaternions)
# ===========================

_EPS = 1e-30

def su2_haar_quaternion(rng, shape):
    x = rng.normal(size=(*shape, 4))
    x /= (np.linalg.norm(x, axis=-1, keepdims=True) + _EPS)
    x = np.where(x[..., :1] < 0, -x, x)
    return x

def quat_mul(a, b):
    aw, ax, ay, az = np.moveaxis(a, -1, 0)
    bw, bx, by, bz = np.moveaxis(b, -1, 0)
    return np.stack([
        aw*bw - ax*bx - ay*by - az*bz,
        aw*bx + ax*bw + ay*bz - az*by,
        aw*by - ax*bz + ay*bw + az*bx,
        aw*bz + ax*by - ay*bx + az*bw
    ], axis=-1)

def quat_unit(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + _EPS)

def su2_brownian_step(rng, U, t):
    v = rng.normal(size=U.shape[:-1] + (3,)) * math.sqrt(max(t, 0.0))
    r = np.linalg.norm(v, axis=-1, keepdims=True)
    axis = v / (r + _EPS)
    q = np.concatenate([
        np.cos(r),
        np.sin(r) * axis
    ], axis=-1)
    q = quat_unit(q)
    return quat_unit(quat_mul(q, U))

# ===========================
# DROP-IN HOOKS
# ===========================

def sample_nu(rng, n_samples, n_links):
    return su2_haar_quaternion(rng, (n_samples, n_links))

def step_K(rng, X, tK):
    return su2_brownian_step(rng, X, tK)

def step_P(rng, X, tP):
    return su2_brownian_step(rng, X, tP)

# ===========================
# Feature map φ
# ===========================

class FeatureMap:
    def __init__(self, rng, n_links, m, scale):
        self.W = rng.normal(size=(m, n_links, 4)) * scale
        self.b = rng.normal(size=(m,)) * (0.1 * scale)

    def phi(self, X):
        return np.einsum("i l d, n l d -> n i", self.W, X) + self.b

# ===========================
# Matrix builders
# ===========================

def sym_outer_mean(D):
    return (D.T @ D) / D.shape[0]

def build_A(PhiX, PhiY, mu):
    DX = (PhiX - mu) - (PhiY - mu)
    return 0.5 * sym_outer_mean(DX)

def build_V(PhiX, mu):
    DX = PhiX - mu
    return sym_outer_mean(DX)

def min_gen_eig(A, B, ridge):
    B = 0.5*(B + B.T) + ridge*np.eye(B.shape[0])
    A = 0.5*(A + A.T)
    L = np.linalg.cholesky(B)
    Linv = np.linalg.inv(L)
    C = Linv @ A @ Linv.T
    return float(np.min(np.linalg.eigvalsh(0.5*(C + C.T))))

# ===========================
# RUN
# ===========================

def run(cfg):
    rng = np.random.default_rng(cfg["seed"])

    X  = sample_nu(rng, cfg["n_samples"], cfg["n_links"])
    YK = step_K(rng, X, cfg["tK"])
    YP = step_P(rng, X, cfg["tP"])

    fmap = FeatureMap(rng, cfg["n_links"], cfg["m_feats"], cfg["scale"])

    PhiX  = fmap.phi(X)
    PhiYK = fmap.phi(YK)
    PhiYP = fmap.phi(YP)

    mu = PhiX.mean(axis=0)

    AK = build_A(PhiX, PhiYK, mu)
    AP = build_A(PhiX, PhiYP, mu)
    V  = build_V(PhiX, mu)

    theta_hat = min_gen_eig(AK, AP, cfg["ridge"])
    c_hat     = min_gen_eig(AK, V,  cfg["ridge"])

    print("=== FINAL BOSS BRIDGE ESTIMATE ===")
    print(f"theta_hat (K vs P) : {theta_hat}")
    print(f"c_hat     (K gap)  : {c_hat}")
    print("---")
    print(cfg)

run(CONFIG)


=== FINAL BOSS BRIDGE ESTIMATE ===
theta_hat (K vs P) : 0.2400693454401243
c_hat     (K gap)  : 0.022674955254425092
---
{'seed': 0, 'n_links': 64, 'n_samples': 20000, 'm_feats': 256, 'tK': 0.02, 'tP': 0.05, 'scale': 1.0, 'ridge': 1e-06, 'chunks': 8}


In [ ]:
#!/usr/bin/env python3
import math
import numpy as np

CONFIG = dict(
    seed=0,
    n_links=64,
    n_samples=20000,
    m_feats=256,
    tK=0.02,
    tP=0.05,
    scale=1.0,
    ridge=1e-6,
    chunks=8,
)

_EPS = 1e-30

def su2_haar_quaternion(rng, shape):
    x = rng.normal(size=(*shape, 4))
    x /= (np.linalg.norm(x, axis=-1, keepdims=True) + _EPS)
    x = np.where(x[..., :1] < 0, -x, x)
    return x

def quat_mul(a, b):
    aw, ax, ay, az = np.moveaxis(a, -1, 0)
    bw, bx, by, bz = np.moveaxis(b, -1, 0)
    return np.stack([
        aw*bw - ax*bx - ay*by - az*bz,
        aw*bx + ax*bw + ay*bz - az*by,
        aw*by - ax*bz + ay*bw + az*bx,
        aw*bz + ax*by - ay*bx + az*bw
    ], axis=-1)

def quat_unit(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + _EPS)

def su2_brownian_step(rng, U, t):
    v = rng.normal(size=U.shape[:-1] + (3,)) * math.sqrt(max(t, 0.0))
    r = np.linalg.norm(v, axis=-1, keepdims=True)
    axis = v / (r + _EPS)
    q = np.concatenate([np.cos(r), np.sin(r) * axis], axis=-1)
    q = quat_unit(q)
    return quat_unit(quat_mul(q, U))

# --- DROP-IN HOOKS ---
def sample_nu(rng, n_samples, n_links):
    return su2_haar_quaternion(rng, (n_samples, n_links))

def step_K(rng, X, tK):
    return su2_brownian_step(rng, X, tK)

def step_P(rng, X, tP):
    return su2_brownian_step(rng, X, tP)

class FeatureMap:
    def __init__(self, rng, n_links, m, scale):
        self.W = rng.normal(size=(m, n_links, 4)) * scale
        self.b = rng.normal(size=(m,)) * (0.1 * scale)
    def phi(self, X):
        return np.einsum("i l d, n l d -> n i", self.W, X) + self.b

def sym_outer_mean(D):
    return (D.T @ D) / D.shape[0]

def build_A(PhiX, PhiY, mu):
    DX = (PhiX - mu) - (PhiY - mu)
    return 0.5 * sym_outer_mean(DX)

def build_V(PhiX, mu):
    DX = PhiX - mu
    return sym_outer_mean(DX)

def min_gen_eig(A, B, ridge):
    A = 0.5*(A + A.T)
    B = 0.5*(B + B.T) + ridge*np.eye(B.shape[0])
    L = np.linalg.cholesky(B)
    Linv = np.linalg.inv(L)
    C = Linv @ A @ Linv.T
    C = 0.5*(C + C.T)
    return float(np.min(np.linalg.eigvalsh(C)))

def compute_theta_c(PhiX, PhiYK, PhiYP, ridge):
    mu = PhiX.mean(axis=0)
    AK = build_A(PhiX, PhiYK, mu)
    AP = build_A(PhiX, PhiYP, mu)
    V  = build_V(PhiX, mu)
    theta = min_gen_eig(AK, AP, ridge)
    c     = min_gen_eig(AK, V,  ridge)
    return theta, c

def run(cfg):
    rng = np.random.default_rng(cfg["seed"])

    X  = sample_nu(rng, cfg["n_samples"], cfg["n_links"])
    YK = step_K(rng, X, cfg["tK"])
    YP = step_P(rng, X, cfg["tP"])

    fmap = FeatureMap(rng, cfg["n_links"], cfg["m_feats"], cfg["scale"])
    PhiX  = fmap.phi(X)
    PhiYK = fmap.phi(YK)
    PhiYP = fmap.phi(YP)

    theta_full, c_full = compute_theta_c(PhiX, PhiYK, PhiYP, cfg["ridge"])

    # chunk stability
    N = PhiX.shape[0]
    idx = np.arange(N)
    rng.shuffle(idx)
    splits = np.array_split(idx, max(1, int(cfg["chunks"])))

    thetas = []
    cs = []
    for s in splits:
        if s.size < max(512, cfg["m_feats"] + 16):
            continue
        th, cc = compute_theta_c(PhiX[s], PhiYK[s], PhiYP[s], cfg["ridge"])
        thetas.append(th); cs.append(cc)

    thetas = np.array(thetas, dtype=np.float64)
    cs     = np.array(cs, dtype=np.float64)

    print("=== FINAL BOSS BRIDGE ESTIMATE (FULL) ===")
    print(f"theta_hat (K vs P) : {theta_full}")
    print(f"c_hat     (K gap)  : {c_full}")
    print("=== STABILITY (CHUNKS) ===")
    print(f"chunks_used        : {thetas.size}")
    if thetas.size:
        print(f"theta min/med/mean/std : {thetas.min()}  {np.median(thetas)}  {thetas.mean()}  {thetas.std()}")
        print(f"c     min/med/mean/std : {cs.min()}      {np.median(cs)}      {cs.mean()}      {cs.std()}")
    print("---")
    print(cfg)

run(CONFIG)


=== FINAL BOSS BRIDGE ESTIMATE (FULL) ===
theta_hat (K vs P) : 0.2400693454401243
c_hat     (K gap)  : 0.022674955254425092
=== STABILITY (CHUNKS) ===
chunks_used        : 8
theta min/med/mean/std : 0.15573612171981122  0.15904178030040683  0.1586567771641305  0.0020822887237770316
c     min/med/mean/std : 0.012450043073934453      0.012746693000557086      0.01278737571052261      0.0002524107401363786
---
{'seed': 0, 'n_links': 64, 'n_samples': 20000, 'm_feats': 256, 'tK': 0.02, 'tP': 0.05, 'scale': 1.0, 'ridge': 1e-06, 'chunks': 8}
